# PN34 remaining-fill rank budget

This executed notebook audits the frozen PN34 bridge. It reads sealed predictions and independently validated truth; it does not refit the coordinate.


In [1]:
from pathlib import Path
import csv, hashlib, json, math
HERE = Path(r'F:\SystemFormulaFolder\GIT\ARA-GIT\analysis\primes')
results = json.loads((HERE/'PN34_FILL_RANK_BUDGET_RESULTS.json').read_text())
validation = json.loads((HERE/'PN34_FILL_RANK_BUDGET_VALIDATION.json').read_text())
primary = json.loads((HERE/'PN34_FILL_RANK_BUDGET_PRIMARY.json').read_text())
print('Test:', results['test_id'])
print('Sealed rows:', primary['row_count'])
print('Prediction hash valid:', hashlib.sha256((HERE/primary['prediction_file']).read_bytes()).hexdigest() == primary['prediction_sha256'])


Test: PN34/FILL-RANK-BUDGET/v1
Sealed rows: 6000
Prediction hash valid: True


In [2]:
for row in results['cohorts']:
    print(row['cohort'],
          'x_B=', round(row['remaining_fill_x'], 6),
          'top1 predicted/observed=', f"{row['predicted_top1']:.4%}/{row['observed_top1']:.4%}",
          'top2=', f"{row['predicted_top2']:.4%}/{row['observed_top2']:.4%}",
          'top3=', f"{row['predicted_top3']:.4%}/{row['observed_top3']:.4%}")


low x_B= 0.215884 top1 predicted/observed= 92.7911%/92.8500% top2= 99.4803%/99.4500% top3= 99.9625%/99.9500%
middle x_B= 0.158364 top1 predicted/observed= 94.6594%/95.4500% top2= 99.7148%/99.7000% top3= 99.9848%/100.0000%
high x_B= 0.134089 top1 predicted/observed= 95.4592%/95.2500% top2= 99.7938%/99.6500% top3= 99.9906%/99.9000%


In [3]:
for row in results['cohorts']:
    assert all(row['calibration_passes'])
    assert all(row['budget_passes'])
print('Calibration checks:', 9, '/', 9)
print('Budget checks:', 6, '/', 6)
print('Predicted order:', results['predicted_scale_order'])
print('Observed order:', results['observed_scale_order'])
print('Direction endpoint:', 'PASS' if results['direction_pass'] else 'FAIL')


Calibration checks: 9 / 9
Budget checks: 6 / 6
Predicted order: ['low', 'middle', 'high']
Observed order: ['low', 'high', 'middle']
Direction endpoint: FAIL


In [4]:
for name, values in results['benchmark_top1'].items():
    print(name, 'Brier=', f"{values['brier']:.9f}", 'log loss=', f"{values['log_loss']:.9f}")
fill = results['benchmark_top1']['fill_prior']['log_loss']
flat = results['benchmark_top1']['flat_pn26_prior']['log_loss']
print('Fill log-loss improvement over flat:', f"{(flat-fill)/flat:.3%}")


fill_prior Brier= 0.051709490 log loss= 0.211445496
flat_pn26_prior Brier= 0.051855083 log loss= 0.212766751
conditional_pnt_prior Brier= 0.058190225 log loss= 0.246562071
Fill log-loss improvement over flat: 0.621%


In [5]:
scientific = {'registered_calibration_thresholds_pass', 'registered_rank_budgets_pass', 'registered_scale_direction_pass'}
implementation = {k:v for k,v in validation['checks'].items() if k not in scientific}
assert all(implementation.values())
assert validation['checks']['registered_scale_direction_pass'] is False
print('Implementation/reconstruction checks:', sum(implementation.values()), '/', len(implementation), 'PASS')
print('Formal verdict: PARTIAL SUPPORT because the registered scale-order endpoint failed.')
print('Boundary:', results['scientific_boundary'])


Implementation/reconstruction checks: 19 / 19 PASS
Formal verdict: PARTIAL SUPPORT because the registered scale-order endpoint failed.
Boundary: The fill coordinate calibrates a population rank budget. Because it is constant inside each cohort, it supplies no within-cohort rule for identifying which first quiet candidate is composite.


## Plain-language result

The untested Phase B gate density predicted how often PN26's first, second and third quiet readings would be needed. All nine calibration tolerances passed on 6,000 fresh anchors. The exact middle-to-high ordering did not, so this is partial support for a population rank budget, not a rule that identifies the individual miss.
